In [ ]:
########################## import modules ###############################

import numpy as np
import os
import sys

import netCDF4 as nc
import linecache as lc
import pandas as pd
import time
from scipy import io as sio
import progressbar
import torch
from shutil import copyfile
import json
import torch
import ipdb
from pyitlib import discrete_random_variable as drv
from multiprocessing import cpu_count, Pool
import tqdm
from numpy import linalg as LA


# import Src
sys.path.append("Src")
from Py_Fun import Project, Print_test, IRI_obj, network_develop

if not os.path.isdir('Projects/'):
    os.mkdir('Projects')

########################## Read Configurations ###############################
#%matplotlib notebook

# Read Config file
with open('Config.json') as json_data_file:
    Config = json.load(json_data_file)

X_select_keys = [key for key in Config['X_select'].keys()]
X_select_values = [value for value in Config['X_select'].values()]

Y_select_keys = [key for key in Config['Y_select'].keys()]
Y_select_values = [value for value in Config['Y_select'].values()]

Stations = [key for key in Config['Test_num'].keys()]
Test_nums = [value for value in Config['Test_num'].values()]

Format_keys = [key for key in Config['Figure_format'].keys()]
Format_values = [value for value in Config['Figure_format'].values()]

# Read Global Coefficients
#with open('Global.json') as Global_coef:
#    Global = json.load(Global_coef)

# Read Project name, ISR datasets and thresholds for preprocessing
Project_name = Config['Names']['Project']
ISR_name = Config['Names']['ISR']
ISR_range = Config['Para_range']

# Read which variable has been picked
index_X = []
index_Y = []


#ipdb.set_trace()

for i in range(len(X_select_keys)):
    if X_select_values[i]:
        #ipdb.set_trace()
        index_X.append(i)
for i in range(len(Y_select_keys)):
    if Y_select_values[i]:
        index_Y.append(i)

########################## Project ###############################

project = Project(Project_name)
project.create()

# remove the project
#project.remove()

Print_test(ISR_range)
time.sleep(1)

num=10
num_init =0

X = []
Y = []
for i in range(num_init,num_init+num):
    print(['data/XY_'+str(i)+'.mat'])
    temp = []
    #print(sio.loadmat(['data/XY_'+str(i)+'.mat']))
    temp = sio.loadmat('data/XY_'+str(i)+'.mat')
    X_temp = temp['X']
    Y_temp = temp['Y']
    Ref_temp = temp['X_ref']
    #import ipdb; ipdb.set_trace()
    if i==num_init:
        X = X_temp
        Y = Y_temp
        Ref = Ref_temp
    else:
        X = np.vstack([X, X_temp])
        Y = np.vstack([Y, Y_temp])
        Ref = np.vstack([Ref, Ref_temp])

    print(X.shape)
    print(Y.shape)

#print(np.argwhere(np.isnan(X)))
#print(~np.isnan(X[7]))

if Config['Flags']['Preprocess']:

    # Read and precprocess ISR data
    Out = project.Preprocess(X.T,Y.T, Ref.T)
    X = Out[0]
    Y = Out[1]
    Ref = Out[2]
    #import ipdb; ipdb.set_trace()
    index_X = np.asarray(index_X).squeeze()
    index_Y = np.asarray(index_Y).squeeze()

    Print_test(index_Y)
    Print_test(Y.shape)

    # Variable Selection
    X_t = X[index_X,:]
    Y_t = Y[index_Y,:]

    #index = (~np.isnan(X[:,7]) & (Y[:,1]>=1e4) & (X[:,0]>200))
    #X_t = X[index,:]
    #Y_t = Y[index,1]

    #print(np.argwhere(np.isnan(X_temp)).shape)

    #import ipdb; ipdb.set_trace()


    # CMI
    Mul_info = project.CMI(X,Y)

    # Normlisation
    X,Y = project.Normlise(X_t, Y_t)

    Y = Y.unsqueeze(0)

    _ , _ = project.Dataset_create(X.T, Y.T, Ref.T)
    #Print_test(Out_train)
    #Print_test(Out_test)
    time.sleep(.5)

########################## Modelling ##################################
# to make it simple, change index_Y.shape to 1
if Config['Flags']['Modelling']:
    #Print_test(len(index_Y))
    net = project.Modelling(len(index_X), 1)
    Print_test(net)
    time.sleep(.5)

    print(X.shape)
    print(Y.shape)
    
########################## Modelling ##################################
# to make it simple, change index_Y.shape to 1
if Config['Flags']['Model_Analysis']:
    #Print_test(len(index_Y))
    RMSD = project.Model_analysis()


########################## Statistical Accuracy ##################################
if Config['Flags']['Accuracy']:

    Acc = np.zeros(len(Stations))
    for i in range(len(Stations)):
        Acc[i] = project.Accuracy(i)
    Print_test(Acc)
    #sio.savemat()
    time.sleep(.5)

########################## Modelling Comparison ##################################
"""
Te Model results comparison

TBT-2012:
DNN model:
COSMIC observation:

"""
if Config['Flags']['Comparison']:

    num = range(Config['IRI_num'])
    #import ipdb; ipdb.set_trace()
    X_t = X_t[:,num]
    Y_t = Y_t[num]
    Ref = Ref[:,num]

    Ne_DNN = project.Predict(X_t, index_X).squeeze()

    #ipdb.set_trace()

    cores = cpu_count()
    p = Pool(cores)
    Print_test(cores)


     #no CPU calculation

    progress = progressbar.ProgressBar()

    a = IRI_obj(Ref)

    

    print('Start to generate Te datasets from IRI-2016...')
    time.sleep(.5)

    #import ipdb; ipdb.set_trace()
    '''
    Te_noCPU = np.zeros([2,X_t.shape[1]])
    for i in progress(range(X_t.shape[1])):
        Te_noCPU[:,i] = a.Ne_IRI_Read(i)
        print(a.Ne_IRI_Read(i))
    '''

    Ne_CPU = []

    for t in tqdm.tqdm(p.imap_unordered(a.Ne_IRI_Read, range(Ref.shape[1])) \
                       , total=Ref.shape[1]):
        Ne_CPU.append(t)

    #Te = p.map(a.Te_IRI_Read, progress(range(X_t.shape[1]))) #X_t.shape[1]-130000)
    Print_test(np.asarray(Ne_CPU))
    Ne = np.asarray(Ne_CPU).squeeze()

    Ne_IRI = Ne/1e6

    index = np.where((Ne_IRI>Config['Para_range']['Ne'][0]) \
                    & (Ne_IRI<Config['Para_range']['Ne'][1]))


    #ipdb.set_trace()
    # save Te results for all

    sio.savemat('Projects/'+Project_name+'/Data/Output/Ne_outputs.mat', \
                {'Ne_IRI':Ne_IRI[index], \
                 'Ne_RO':Y_t[index], \
                 'Ne_DNN':Ne_DNN[index], \
                 'Ne_ref':Ref[:,index]})
    #ipdb.set_trace()
        #project.move(Stations[i]+'_Te_outputs.mat',\
        #             'Data/', 'Data/Output/')
    acc_IRI = LA.norm(Ne_IRI[index]-Y_t[index], ord=2)/np.sqrt(Y_t[index].shape)
    acc_model = LA.norm(Ne_DNN[index]-Y_t[index], ord=2)/np.sqrt(Y_t[index].shape)

    print('acc_IRI = ', acc_IRI)
    print('acc_model = ', acc_model)
    print('\r')
    time.sleep(.5)

########################## Figures Generation ##################################

if Config['Flags']['Generation']:

    project.Show_case()


The project 'test_CMI_10m' does exist. Would you like to overwrite it? (Y/N) y
The project 'test_CMI_10m' has been overwritten.
['data/XY_0.mat']
(1000246, 15)
(1000246, 2)
['data/XY_1.mat']
(2000254, 15)
(2000254, 2)
['data/XY_2.mat']
(3000334, 15)
(3000334, 2)
['data/XY_3.mat']
(4000661, 15)
(4000661, 2)
['data/XY_4.mat']
(5000881, 15)
(5000881, 2)
['data/XY_5.mat']
(6000966, 15)
(6000966, 2)
['data/XY_6.mat']
(7001403, 15)
(7001403, 2)
['data/XY_7.mat']
(8001754, 15)
(8001754, 2)
['data/XY_8.mat']
(9001783, 15)
(9001783, 2)
['data/XY_9.mat']
(10002219, 15)
(10002219, 2)


Src/Py_Fun.py:259: RuntimeWarning: invalid value encountered in greater
  (Longitude>Config['Para_range']['Longitude'][0]) & (Longitude<Config['Para_range']['Longitude'][1]))#and X[0,:]<=500
Src/Py_Fun.py:259: RuntimeWarning: invalid value encountered in less
  (Longitude>Config['Para_range']['Longitude'][0]) & (Longitude<Config['Para_range']['Longitude'][1]))#and X[0,:]<=500


Given Altitude, MI(CMI) between Ne and Altitude is 0.0
Given Altitude, MI(CMI) between Ne and Latitude is 0.22
Given Altitude, MI(CMI) between Ne and Longitude is 0.22
Given Altitude, MI(CMI) between Ne and Azi is 0.22
Given Altitude, MI(CMI) between Ne and DST is 0.22
Given Altitude, MI(CMI) between Ne and AE is 0.22
Given Altitude, MI(CMI) between Ne and AP is 0.2
Given Altitude, MI(CMI) between Ne and F107 is 0.22
Given Altitude, MI(CMI) between Ne and Kp is 0.2
Given Altitude, MI(CMI) between Ne and DoY is 0.22
Given Altitude, MI(CMI) between Ne and UT is 0.22
Given Altitude, MI(CMI) between Ne and hmF2 is 0.22
Given Altitude, MI(CMI) between Ne and NmF2 is 0.22
Given Altitude, MI(CMI) between Ne and VSH is 0.22
Given Latitude, MI(CMI) between Ne and Altitude is 0.1
Given Latitude, MI(CMI) between Ne and Latitude is 0.0
Given Latitude, MI(CMI) between Ne and Longitude is 0.1
Given Latitude, MI(CMI) between Ne and Azi is 0.1
Given Latitude, MI(CMI) between Ne and DST is 0.1
Given La